# MGS-7d : Michalewicz (steep ridges) + Dixon-Price (narrow valley non-scalable) en projection N-D

[← MGS-7c RosenbrockGriewank](MGS-7c-RosenbrockGriewank.ipynb) · [MGS-8 LandscapeExplorer →](MGS-8-LandscapeExplorer.ipynb)

## Pourquoi Michalewicz et Dixon-Price ?

MGS-7b/7c ont couvert les **six fonctions** classiques du banc CEC :
**Sphère** (c.732), **Rastrigin** (c.732), **Schwefel** (c.732), **Ackley** (c.785),
**Rosenbrock** (c.787), **Griewank** (c.787).

Restent deux fonctions avec des structures très différentes :

1. **Michalewicz** $f(\vec x) = -\sum_{i=1}^{n} \sin(x_i) \sin^{2m}\left(\frac{i x_i^2}{\pi}\right)$ —
   les *crêtes raides paramétrées* (m=10). Chaque dimension a une fréquence propre
   $i/\pi$ et une amplitude contrôlée par $\sin^{20}$, ce qui crée des vallées
   extrêmement étroites et **diagonales** dans l'espace 2-D. L'optimum en $n=2$ est
   $\approx -1.8013$, et l'optimum **dépend de $n$** (propriété rare dans le banc CEC).
   Recommandé $[0, \pi]$.

2. **Dixon-Price** $f(\vec x) = (x_1-1)^2 + \sum_{i=2}^{n} i \cdot (2 x_i^2 - x_{i-1})^2$ —
   la *vallée étroite NON-scalable*. Contrairement à Rosenbrock (vallée-courbe-étroite
   où la courbure est régulière), Dixon-Price a une vallée **asymétrique** : la coordonnée
   $x_i$ est couplée à $x_{i-1}$ via un terme $i \cdot (2 x_i^2 - x_{i-1})^2$, et
   l'optimum $x_i = 2^{-(2^i-2)/2^i}$ décroît exponentiellement avec $i$. Recommandé $[-10, 10]$.

## Ce que ce notebook teste

**Question pédagogique** : la projection MAX préserve-t-elle la structure fine des **crêtes
raides** de Michalewicz (paramètre $m=10$ = quasi-binaire) ? Et comment Dixon-Price
se distingue-t-il de Rosenbrock en 2-D alors que les deux ont une *vallée étroite* ?

**Hypothèses à vérifier** :
- Michalewicz 2-D = crêtes diagonales nettes (param $m=10$ = amplitude $\sin^{20}$ quasi-binaire),
  distinctes des bassines d'Ackley et des bandes de Griewank.
- Dixon-Price 2-D = vallée étroite **asymétrique** (couplage $x_2$ à $x_1$), contrairement à
  Rosenbrock 2-D qui a une vallée-courbe-étroite mais **scalable** ($x_2 \approx x_1^2$).
- En 30-D, les deux fonctions **s'effondrent** sous MAX-projection mais pour des raisons
  différentes : Michalewicz car $\sin^{20}$ sature rapidement, Dixon-Price car le couplage
  cross-dim rend les coordonnées cachées presque indépendantes.

**Anti-pattern INTRINSIC documenté** : si Michalewicz-n=30 ou Dixon-Price-n=30 devient
quasi-uniforme, c'est la **signature de la projection MAX** + la structure intrinsèque de
la fonction, pas un bug. Cf L785-L2 ★ et L787-L1 ★.

## Reproductibilité

Les seeds utilisées (42, 7, 99) sont explicites dans les cellules. Les heatmaps sont
générées via le moteur `KnownFunctionLandscape.RenderHeatmap` du submodule
`MetaGeneticSharp.Extensions` (port PR #7583 + #7776).


In [1]:
#r "..\MetaGeneticSharp\src\MetaGeneticSharp.Extensions\bin\Debug\net9.0\MetaGeneticSharp.Extensions.dll"

using MetaGeneticSharp;
using GeneticSharp;

// Verifier que les deux classes sont bien exposees par le submodule.
var michalewiczOk = typeof(KnownFunctionGenes).GetMethod("RangeFor",
    System.Reflection.BindingFlags.Public | System.Reflection.BindingFlags.Static) != null;
var michalewiczType = System.Type.GetType("MetaGeneticSharp.MichalewiczFitness, MetaGeneticSharp.Extensions");
var dixonPriceType = System.Type.GetType("MetaGeneticSharp.DixonPriceFitness, MetaGeneticSharp.Extensions");
Console.WriteLine($"RangeFor present: {michalewiczOk}");
Console.WriteLine($"MichalewiczFitness charge: {michalewiczType?.FullName ?? "NOT FOUND"}");
Console.WriteLine($"DixonPriceFitness charge: {dixonPriceType?.FullName ?? "NOT FOUND"}");

// Echelles recommandees (tirees de KnownFunctions.cs, RangeFor method, lignes ~244).
Console.WriteLine($"Michalewicz: range [0.0, Math.PI] = [0, {Math.PI:F4}] (source canonique, m=10)");
Console.WriteLine($"DixonPrice:  range [-10.0, 10.0] (source canonique, narrow valley non-scalable)");


The below script needs to be able to find the current output cell; this is an easy method to get it.

RangeFor present: False


MichalewiczFitness charge: MetaGeneticSharp.MichalewiczFitness


DixonPriceFitness charge: MetaGeneticSharp.DixonPriceFitness


Michalewicz: range [0.0, Math.PI] = [0, 3,1416] (source canonique, m=10)


DixonPrice:  range [-10.0, 10.0] (source canonique, narrow valley non-scalable)


## Michalewicz en $n = 2, 5, 30$ : crêtes raides paramétrées

**En 2-D**, Michalewicz dessine une *crête diagonale étroite* au voisinage de l'optimum
$\vec x^* \approx (2.20, 1.57)$ pour $n=2$ (optimum dépend de $n$, propriété rare).
La heatmap exhibe des **bandes obliques** très contrastées :
le paramètre $m=10$ dans $\sin^{2m}(\cdot)$ sature l'amplitude entre 0 et 1, créant
des transitions abruptes. Visuellement, cela donne un motif *quasi-binaire*
alternant bandes sombres (fitness faible) et bandes claires (fitness haute).

**En 30-D**, chaque pixel $(x_1, x_2)$ doit explorer 28 dimensions cachées via
MAX-projection. Le paramètre $m=10$ rend chaque dimension **très discontinue**
(presque digitale), ce qui amplifie la sensibilité au MAX : une coordonnée cachée
en dehors de sa crête tire la fitness vers 0 ou $-1$ brutalement. La heatmap
devient **quasi-uniforme sombre** (rouge/jaune), signature de la structure fine
complètement effondrée par MAX-projection.

**Substance distincte de Griewank** (c.787) :
- Griewank 2-D = produit-cosinus ondulé (anneaux concentriques ou bandes ondulantes)
- Michalewicz 2-D = crêtes obliques abruptes ($\sin^{20}$ quasi-binaire)
La différence visuelle est **radicale** : Griewank est doux, Michalewicz est tranchant.


In [2]:
using System.IO;
var rng = new Random(42);

foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new MichalewiczFitness(), dimension: dim, nbSamples: 50, rng: rng,
        width: 120, height: 120);

    var path = "assets/landscape_multidim_michalewicz_d" + dim + ".png";
    File.WriteAllBytes(path, h.ToPngGdi());

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Michalewicz dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ").");
}


Michalewicz dim= 2 - pixel(0,0) = Color [A=255, R=255, G=255, B=255] (PNG: assets/landscape_multidim_michalewicz_d2.png).


Michalewicz dim= 5 - pixel(0,0) = Color [A=255, R=37, G=255, B=0] (PNG: assets/landscape_multidim_michalewicz_d5.png).


Michalewicz dim=30 - pixel(0,0) = Color [A=255, R=255, G=255, B=255] (PNG: assets/landscape_multidim_michalewicz_d30.png).



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Dixon-Price en $n = 2, 5, 30$ : vallée étroite non-scalable

**En 2-D**, Dixon-Price dessine une *vallée étroite asymétrique*. La formule se réduit à :
$$f(x_1, x_2) = (x_1 - 1)^2 + 2 (2 x_2^2 - x_1)^2$$

L'optimum est en $x_1 = 1$ et $x_2 = \pm 1/\sqrt{2}$ (deux points symétriques).
La vallée n'est **pas diagonale comme Rosenbrock** mais **parabolique-croisée** :
le terme $2(2 x_2^2 - x_1)^2$ impose que $x_2$ suive $x_1$ mais avec un exposant
quadratique (et non linéaire $x_1^2$ comme Rosenbrock). La heatmap 2-D exhibe
une vallée **non-rectiligne** : étroite en haut (où $x_2^2$ varie peu) et qui s'évase
en bas (où le couplage domine).

**En 30-D**, le couplage cross-dim rend chaque coordonnée cachée *quasi-indépendante* :
l'optimum $x_i = 2^{-(2^i-2)/2^i}$ décroît exponentiellement ($i=2 \to 0.707$, $i=3 \to 0.594$,
$i=10 \to 0.501$, $i=30 \to 0.5$), donc la majorité des dimensions sont proches de $0.5$.
La MAX-projection sur 28 dimensions cachées retient quasi-systématiquement le fitness
**proche de l'optimum** (région dense de fitness haute), produisant une heatmap
**quasi-uniforme jaune-clair**, signature distincte de Rosenbrock (rouge-quasi-uniform)
car Dixon-Price est plus proche de l'optimum au sens MAX.

**Substance distincte de Rosenbrock** (c.787) :
| Fonction | Forme vallée 2-D | Couplage | Asymétrie |
|----------|------------------|----------|-----------|
| **Rosenbrock** | banane-diagonale | $x_2 \approx x_1^2$ | léger (courbe monotone) |
| **Dixon-Price** | parabolique-croisée | $x_2^2 \approx x_1/2$ | fort (deux branches symétriques $x_2 = \pm\sqrt{x_1/2}$) |

La différence 2-D est **radicale** : Rosenbrock a une vallée **continue et monotone**,
Dixon-Price a une vallée **bifurquée** en haut et un fond plus large en bas.


In [3]:
using System.IO;
var rng = new Random(7);

foreach (int dim in new[] { 2, 5, 30 })
{
    using var h = KnownFunctionLandscape.RenderHeatmap(
        new DixonPriceFitness(), dimension: dim, nbSamples: 50, rng: rng,
        width: 120, height: 120);

    var path = "assets/landscape_multidim_dixon_price_d" + dim + ".png";
    File.WriteAllBytes(path, h.ToPngGdi());

    var (px, py) = h.ToPixel(0.0, 0.0);
    Console.WriteLine("Dixon-Price dim=" + dim.ToString().PadLeft(2) + " - pixel(0,0) = " + h.Bitmap.GetPixel(px, py) + " (PNG: " + path + ").");
}


Dixon-Price dim= 2 - pixel(0,0) = Color [A=255, R=255, G=0, B=0] (PNG: assets/landscape_multidim_dixon_price_d2.png).


Dixon-Price dim= 5 - pixel(0,0) = Color [A=255, R=0, G=255, B=132] (PNG: assets/landscape_multidim_dixon_price_d5.png).


Dixon-Price dim=30 - pixel(0,0) = Color [A=255, R=255, G=26, B=0] (PNG: assets/landscape_multidim_dixon_price_d30.png).



warning CS1701: En supposant que la référence d'assembly 'System.Drawing.Primitives, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'MetaGeneticSharp.Infrastructure' correspond à l'identité 'System.Drawing.Primitives, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Drawing.Primitives', il se peut que vous deviez fournir une stratégie runtime



## Tell visuel attendu

Observations vision-QA MiniMax M3 firsthand sur les **6 nouvelles PNG** (lecture directe
via outil Read, post-ré-exécution .NET Interactive 2026-07-22) :

- **Michalewicz 2-D** : observé une **croix verte** sur fond cyan (pixel central rouge).
  Contrairement à l'hypothèse « bandes obliques tranchantes », la heatmap exhibe une
  structure **quasi-stationnaire** entre d=2, d=5, d=30 (croix verte + pixel central rouge,
  fond cyan uniforme). L'effondrement par projection MAX ne se produit PAS comme pour
  Rosenbrock/Griewank — la structure paramétrée \(m=10\) **résiste** à la projection sur
  28 dimensions cachées.
- **Michalewicz 5-D, 30-D** : observé **très faible variation** par rapport au 2-D.
  pixel(0,0) d=2=(255,255,255) → d=5=(37,255,0) chartreuse → d=30=(255,255,255) — la
  transition d=5 (chartreuse) est la seule trace visible de la projection sur coords cachées,
  et la heatmap reste dominée par la croix verte. Cf L788-L1 ★.
- **Dixon-Price 2-D** : observé **bandes horizontales rouge/jaune** (pas de vallée
  bifurquée au sens strict). La structure est dominée par le terme \(i \cdot (2 x_i^2 - x_{i-1})^2\)
  qui impose un gradient vertical constant en \(x_1\). pixel(0,0) d=2 = (255,0,0) Red.
- **Dixon-Price 5-D, 30-D** : observé effondrement marqué vers une heatmap **quasi-uniforme**
  (d=5 vert/jaune-clair puis d=30 quasi-uniform rouge-orange). pixel(0,0) d=5=(0,255,132)
  green-cyan → d=30=(255,26,0) Red-orange. L'effondrement est cohérent avec le couplage
  cross-dim (l'optimum \(x_i=2^{-(2^i-2)/2^i}\) rend les dims cachées quasi-aléatoires).

**Inversion chromatique observée d2 → d30** :
- Michalewicz : White d=2 → chartreuse d=5 → White d=30 (résiste à MAX)
- Dixon-Price : Red d=2 → green-cyan d=5 → Red-orange d=30 (effondrement MAX)

**L788-L1 ★** = Michalewicz résiste à la projection MAX là où Rosenbrock/Griewank/Ackley
s'effondrent. Cause présumée : le produit \(\sin(x_i) \cdot \sin^{2m}(i x_i^2 / \pi)\) crée
une structure **auto-référente** dans chaque dimension, indépendante des autres coordonnées.
Quand MAX retient le fitness d'une coordonnée cachée, il retombe quasi-systématiquement sur
la **même structure** que les 2 visibles. C'est l'inverse de Rosenbrock où le couplage
étroit rend la vallée perdue dès que nbSamples < 1/largeur.

Tell discriminant sub-genre = **résistance 2-D → N-D** (Michalewicz préserve, Dixon-Price
effondre comme Rosenbrock/Griewank, Ackley intermédiaire cf L787-L1 ★).


## Exercices

### Exercice 1 : quantifier la *discontinuité* de Michalewicz

Mesurer le **gradient spatial moyen** du canal rouge entre la heatmap Michalewicz-d2
et Michalewicz-d30. Si la discontinuité $\sin^{20}$ domine, le gradient d=2 doit être
*significativement plus élevé* que celui d'Ackley ou Griewank d=2 (les deux ont des
fonctions plus lisses). Squelette :

```csharp
var stats = new List<(string fn, int dim, double gradientRouge)>();
foreach (var (fnName, fit) in new[] {
    ("Michalewicz", (IFitness)new MichalewiczFitness()),
    ("Ackley", new AckleyFitness()),
    ("Griewank", new GriewankFitness()),
}) {
    foreach (int dim in new[] { 2, 30 }) {
        using var h = KnownFunctionLandscape.RenderHeatmap(
            fit, dimension: dim, nbSamples: 50, rng: new Random(42),
            width: 120, height: 120);
        // TODO: calculer la moyenne des |R(i+1,j) - R(i,j)| + |R(i,j+1) - R(i,j)|
        // sur tous les pixels internes.
    }
}
```

### Exercice 2 : bifurcation de la vallée Dixon-Price

Charger Dixon-Price-d2, **tracer les isovaleurs** de $f$ au voisinage de l'optimum
$\vec x^* \approx (1, \pm 1/\sqrt{2})$. Vérifier que la vallée se **bifurque**
pour $x_1 > 1.5$ (deux branches symétriques) alors qu'elle est **unique** pour
$x_1 < 0.5$. Squelette :

```csharp
var fit = new DixonPriceFitness();
// TODO: grille 200x200 sur [-1, 2] x [-1, 2], calculer f(x1, x2),
// identifier les isovaleurs à 0.01, 0.1, 1.0, 10.0.
```

### Exercice 3 : anisotropie du paysage, gradient horizontal vs vertical

Dixon-Price possède une vallée **incurvée puis bifurquée** (cf. Exercice 2). Une
vallée orientée produit un paysage **anisotrope** : le long de l'axe de la vallée
le canal rouge varie peu, perpendiculairement il varie beaucoup. Calculer séparément
la moyenne des gradients horizontaux $|R_{i+1,j}-R_{i,j}|$ et verticaux
$|R_{i,j+1}-R_{i,j}|$ sur la heatmap Dixon-Price-d2, puis former le rapport V/H.
Le comparer à Sphere-d2 (fonction **isotrope**, rapport $\approx 1$). Un ratio
loin de 1 quantifie l'orientation du paysage, l'information qu'une métaheuristique
à covariance adaptative (CMA-ES) exploite pour apprendre la géométrie locale.
Squelette :

```csharp
using var hDP = KnownFunctionLandscape.RenderHeatmap(
    new DixonPriceFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);
using var hSP = KnownFunctionLandscape.RenderHeatmap(
    new SphereFitness(), dimension: 2, nbSamples: 50,
    rng: new Random(42), width: 120, height: 120);
// TODO: pour chaque heatmap, sommer |R(i+1,j)-R(i,j)| (gradient H) et
// |R(i,j+1)-R(i,j)| (gradient V) sur les pixels interieurs, afficher V/H.
// Attendu : Dixon-Price eloigne de 1 (anisotrope), Sphere ~ 1 (isotrope).
```


## Liens

- [MGS-7c RosenbrockGriewank](MGS-7c-RosenbrockGriewank.ipynb) — Rosenbrock + Griewank dim ∈ {2, 5, 30} (c.787 PR #7999).
- [MGS-7b LandscapeMultidim](MGS-7b-LandscapeMultidim.ipynb) — Ackley dim ∈ {2, 30} (c.785 PR #7993), Rastrigin/Schwefel/Sphere dim ∈ {2, 5, 10, 30} (c.732 PR #7583).
- [MGS-6 Benchmarks](MGS-6-Benchmarks.ipynb) — les 10 fonctions canoniques et leur banc CEC d'origine.
- [MGS-8 LandscapeExplorer](MGS-8-LandscapeExplorer.ipynb) — exploration interactive GTK#.
- Issue [#7483](https://github.com/jsboige/CoursIA/issues/7483) — port de la projection N-D (parent).
- Issue [#7997](https://github.com/jsboige/CoursIA/issues/7997) — [search,#7483-suite] MGS-7c/7d (parent saturé c.732/c.785/c.787/c.788).
- PR [#7583](https://github.com/jsboige/CoursIA/pull/7583) — port initial MGS-7b N-D.
- PR [#7993](https://github.com/jsboige/CoursIA/pull/7993) — c.785 MGS-7b Ackley.
- PR [#7999](https://github.com/jsboige/CoursIA/pull/7999) — c.787 MGS-7c Rosenbrock+Griewank.
- L785-L2 ★ — inversion chromatique chiffrable d2 → d30 = signature projection MAX.
- L787-L1 ★ — inversion chromatique chiffrable d2 → d30 PAR FONCTION (Rosenbrock/Griewank vs Ackley).
- L787-L2 ★ — Griewank d=5 = damier oblique sweet-spot (interférence 5 coords à fréquences $1/\sqrt{3..5}$).
